In [8]:
import tempfile
import os
from ansys.aedt.core import Desktop, generate_unique_name
import optuna
import script_otim as script
import logging
import sys
import numpy as np

In [9]:
optuna.logging.get_logger("optuna").addHandler(logging.StreamHandler(sys.stdout))
study_name = "Monopole Optimization 5" 
storage_name = "sqlite:///{}.db".format(study_name)

In [10]:
tmpfold = tempfile.gettempdir()

temp_folder = os.path.join(
    tmpfold,
    generate_unique_name("Antenna")
)

if not os.path.exists(temp_folder):
    os.mkdir(temp_folder)

non_graphical = os.getenv(
    "PYAEDT_NON_GRAPHICAL",
    "True"
).lower() in ("true", "1", "t")

d = Desktop(
    "2025.1",
    non_graphical=non_graphical,
    new_desktop=True
)


PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.18.1.
PyAEDT INFO: Returning found Desktop session with PID 9544!


In [11]:
def objective(trial):
    a1 = trial.suggest_float("a1", 1.0, 1.0)
    a2 = trial.suggest_float("a2", 1.0, 1.0)
    n1 = trial.suggest_int("n1", 1, 100)
    n2 = trial.suggest_float("n2", 1.0, 10.0)
    n3 = trial.suggest_float("n3", 1.0, 10.0)
    n4 = trial.suggest_int("n4", 1, 100)
    perc = trial.suggest_float("perc", 0.2, 0.8)
    r = trial.suggest_int("r", 90, 150)

    df = script.makeAntenna(a1, a2, n1, n2, n3, n4, perc, r, temp_folder)
    s11 = np.asarray(df['|$S_{11}$| [dB]'], dtype=float)

    n = len(s11)
    idx1 = n // 3
    idx2 = 2 * n // 3

    weight = np.zeros(n)
    weight[:idx1] = 1
    weight[idx1:idx2] = 0.9
    weight[idx2:] = 0.7
    
    excess= np.maximum(s11 + 10, 0.0)
    
    weighted_excess = np.average(excess, weights=weight)

    mask = s11 <= -10
    prefix_len = 0
    for ok in mask:
        if ok:
            prefix_len += 1
        else:
            break
    band = prefix_len / len(s11)

    return band, weighted_excess

study = optuna.create_study(
    directions=["maximize", "minimize"],    
    study_name=study_name, 
    storage=storage_name, 
    load_if_exists=True
)



[I 2025-09-15 08:12:41,507] Using an existing study with name 'Monopole Optimization 5' instead of creating a new one.


Using an existing study with name 'Monopole Optimization 5' instead of creating a new one.
Using an existing study with name 'Monopole Optimization 5' instead of creating a new one.


In [ ]:
study.optimize(objective, n_trials=1000)

PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.18.1.
PyAEDT INFO: Returning found Desktop session with PID 9544!
PyAEDT INFO: Project Project3084 has been created.
PyAEDT INFO: No design is present. Inserting a new design.
PyAEDT INFO: Added design 'HFSS_E5Y' of type HFSS.
PyAEDT INFO: Aedt Objects correctly read
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3D Modeler objects parsed. Elapsed time: 0m 0sec
PyAEDT INFO: Deleted 1 Objects: cut.
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m

[I 2025-09-15 08:15:08,821] Trial 2923 finished with values: [0.0, 3.3729785233517733] and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 5, 'n2': 7.693155084578477, 'n3': 2.2045426188436146, 'n4': 28, 'perc': 0.5531568083211025, 'r': 110}.


Trial 2923 finished with values: [0.0, 3.3729785233517733] and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 5, 'n2': 7.693155084578477, 'n3': 2.2045426188436146, 'n4': 28, 'perc': 0.5531568083211025, 'r': 110}.
Trial 2923 finished with values: [0.0, 3.3729785233517733] and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 5, 'n2': 7.693155084578477, 'n3': 2.2045426188436146, 'n4': 28, 'perc': 0.5531568083211025, 'r': 110}.
PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.18.1.
PyAEDT INFO: Returning found Desktop session with PID 9544!
PyAEDT INFO: Project Project3086 has been created.
PyAEDT INFO: No design is present. Inserting a new design.
PyAEDT INFO: Added design 'HFSS_EHU' of type HFSS.
PyAEDT INFO: Aedt Objects correctly read
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Parsing 

[I 2025-09-15 08:18:14,125] Trial 2924 finished with values: [0.0, 6.1949139476615205] and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 68, 'n2': 7.567437494841701, 'n3': 1.0376151878067557, 'n4': 5, 'perc': 0.5531568083211025, 'r': 128}.


Trial 2924 finished with values: [0.0, 6.1949139476615205] and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 68, 'n2': 7.567437494841701, 'n3': 1.0376151878067557, 'n4': 5, 'perc': 0.5531568083211025, 'r': 128}.
Trial 2924 finished with values: [0.0, 6.1949139476615205] and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 68, 'n2': 7.567437494841701, 'n3': 1.0376151878067557, 'n4': 5, 'perc': 0.5531568083211025, 'r': 128}.
PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.18.1.
PyAEDT INFO: Returning found Desktop session with PID 9544!
PyAEDT INFO: Project Project3089 has been created.
PyAEDT INFO: No design is present. Inserting a new design.
PyAEDT INFO: Added design 'HFSS_GG6' of type HFSS.
PyAEDT INFO: Aedt Objects correctly read
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Parsing 

...

Trial 2977 finished with values: [0.05, 5.043509247105793] and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 33, 'n2': 8.712217224342535, 'n3': 4.4826184918502685, 'n4': 28, 'perc': 0.6374839496838001, 'r': 110}.
Trial 2977 finished with values: [0.05, 5.043509247105793] and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 33, 'n2': 8.712217224342535, 'n3': 4.4826184918502685, 'n4': 28, 'perc': 0.6374839496838001, 'r': 110}.
PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.18.1.
PyAEDT INFO: Returning found Desktop session with PID 9544!
PyAEDT INFO: Project Project3204 has been created.
PyAEDT INFO: No design is present. Inserting a new design.
PyAEDT INFO: Added design 'HFSS_QD3' of type HFSS.
PyAEDT INFO: Aedt Objects correctly read
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Parsin

[I 2025-09-15 11:20:41,331] Trial 2978 finished with values: [0.0, 0.23851531128056538] and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 67, 'n2': 9.717984816125238, 'n3': 1.678081519138153, 'n4': 28, 'perc': 0.6583451213622109, 'r': 110}.


Trial 2978 finished with values: [0.0, 0.23851531128056538] and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 67, 'n2': 9.717984816125238, 'n3': 1.678081519138153, 'n4': 28, 'perc': 0.6583451213622109, 'r': 110}.
Trial 2978 finished with values: [0.0, 0.23851531128056538] and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 67, 'n2': 9.717984816125238, 'n3': 1.678081519138153, 'n4': 28, 'perc': 0.6583451213622109, 'r': 110}.
PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.18.1.
PyAEDT INFO: Returning found Desktop session with PID 9544!
PyAEDT INFO: Project Project3206 has been created.
PyAEDT INFO: No design is present. Inserting a new design.
PyAEDT INFO: Added design 'HFSS_T8J' of type HFSS.
PyAEDT INFO: Aedt Objects correctly read
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Parsin

[I 2025-09-15 11:23:12,414] Trial 2979 finished with values: [0.0, 0.2280683499370735] and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 38, 'n2': 9.764674264900336, 'n3': 5.854595807495781, 'n4': 5, 'perc': 0.6606149221073836, 'r': 110}.


Trial 2979 finished with values: [0.0, 0.2280683499370735] and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 38, 'n2': 9.764674264900336, 'n3': 5.854595807495781, 'n4': 5, 'perc': 0.6606149221073836, 'r': 110}.
Trial 2979 finished with values: [0.0, 0.2280683499370735] and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 38, 'n2': 9.764674264900336, 'n3': 5.854595807495781, 'n4': 5, 'perc': 0.6606149221073836, 'r': 110}.
PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.18.1.
PyAEDT INFO: Returning found Desktop session with PID 9544!
PyAEDT INFO: Project Project3208 has been created.
PyAEDT INFO: No design is present. Inserting a new design.
PyAEDT INFO: Added design 'HFSS_UON' of type HFSS.
PyAEDT INFO: Aedt Objects correctly read
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Parsing de

In [ ]:
len(study.trials)

In [ ]:
study.best_params

In [ ]:
study.best_trial

In [ ]:
d.release_desktop()